In [1]:
!mamba install pandas numpy scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.368 seconds
  Name                Version    Build                Channel
---------------------------------------------------------------------------
+ brotli-python       1.2.0      py313ha26e73d_2      emscripten-forge-4x
+ certifi             2026.7.22  pyhd8ed1ab_0         conda-forge
+ charset-normalizer  3.4.9      pyhd8ed1ab_0         conda-forge
+ idna                3.18       pyhcf101f3_0         conda-forge
+ joblib              1.5.3      py313h1804a44_3      emscripten-forge-4x
+ narwhals            2.24.0     pyhcf101f3_0         conda-forge
+ pandas              3.0.5      np23py313h1e705a5_0  emscripten-forge-4x
+ pysocks             1.7.1      py313h1804a44_3      emscripten-forge-4x
+ python-tzdata       2026.3     pyhd8ed1ab_0         conda-forge
+ requests          

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", __import__("sklearn").__version__)
print("DAY 10 ENVIRONMENT READY")

Pandas: 3.0.5
NumPy: 2.5.0
Scikit-learn: 1.9.0
DAY 10 ENVIRONMENT READY


In [3]:
import pandas as pd

train_df = pd.read_csv("train_fraud_balanced.csv")

print("Training data loaded successfully!")
print("Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

Training data loaded successfully!
Shape: (79328, 45)

Columns:
['transaction_id', 'transaction_date', 'customer_id', 'card_type', 'amount', 'merchant_category', 'merchant_country', 'transaction_channel', 'device_type', 'is_international', 'is_card_present', 'transactions_last_24h', 'average_amount_30d', 'risk_score', 'is_fraud', 'transaction_hour', 'transaction_day', 'transaction_month', 'dataset_source', 'sender_account_id', 'receiver_account_id', 'transaction_type', 'sender_balance_before', 'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after', 'customer_age_group', 'customer_city', 'kyc_level', 'login_attempts', 'account_age_days', 'product_category', 'payment_method', 'billing_country', 'shipping_country', 'email_age_days', 'ip_risk_level', 'failed_payment_attempts', 'delivery_speed', 'customer_balance', 'location', 'distance_from_home_km', 'transactions_last_1h', 'new_device', 'new_location']


In [4]:
print("Target column: is_fraud")
print(train_df["is_fraud"].value_counts())

print("\nTarget percentage:")
print(train_df["is_fraud"].value_counts(normalize=True) * 100)

Target column: is_fraud
is_fraud
0    39664
1    39664
Name: count, dtype: int64

Target percentage:
is_fraud
0    50.0
1    50.0
Name: proportion, dtype: float64


In [5]:
features = [
    "amount",
    "is_international",
    "is_card_present",
    "transactions_last_24h",
    "average_amount_30d",
    "risk_score",
    "transaction_hour",
    "transaction_day",
    "transaction_month",
    "sender_balance_before",
    "sender_balance_after",
    "receiver_balance_before",
    "receiver_balance_after",
    "login_attempts",
    "account_age_days",
    "email_age_days",
    "failed_payment_attempts",
    "customer_balance",
    "distance_from_home_km",
    "transactions_last_1h",
    "new_device",
    "new_location"
]

X_train = train_df[features]
y_train = train_df["is_fraud"]

print("Number of features:", len(features))
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

Number of features: 22
X_train shape: (79328, 22)
y_train shape: (79328,)


In [6]:
print("Missing values in training features:")
print(X_train.isnull().sum())

print("\nTotal missing values:", X_train.isnull().sum().sum())

Missing values in training features:
amount                         0
is_international           66803
is_card_present            66803
transactions_last_24h      66803
average_amount_30d         66803
risk_score                     0
transaction_hour               0
transaction_day                0
transaction_month              0
sender_balance_before      61588
sender_balance_after       61588
receiver_balance_before    61588
receiver_balance_after     61588
login_attempts             61588
account_age_days           39703
email_age_days             57443
failed_payment_attempts    57443
customer_balance           52150
distance_from_home_km      52150
transactions_last_1h       52150
new_device                 52150
new_location               52150
dtype: int64

Total missing values: 990491


In [7]:
from sklearn.impute import SimpleImputer

# Binary features
binary_features = [
    "is_international",
    "is_card_present",
    "new_device",
    "new_location"
]

# Numerical features
numeric_features = [
    "amount",
    "transactions_last_24h",
    "average_amount_30d",
    "risk_score",
    "transaction_hour",
    "transaction_day",
    "transaction_month",
    "sender_balance_before",
    "sender_balance_after",
    "receiver_balance_before",
    "receiver_balance_after",
    "login_attempts",
    "account_age_days",
    "email_age_days",
    "failed_payment_attempts",
    "customer_balance",
    "distance_from_home_km",
    "transactions_last_1h"
]

# Impute numerical features using median
numeric_imputer = SimpleImputer(strategy="median")
X_train[numeric_features] = numeric_imputer.fit_transform(
    X_train[numeric_features]
)

# Impute binary features using most frequent value
binary_imputer = SimpleImputer(strategy="most_frequent")
X_train[binary_features] = binary_imputer.fit_transform(
    X_train[binary_features]
)

print("Missing values handled successfully!")
print("Total missing values:", X_train.isnull().sum().sum())

Missing values handled successfully!
Total missing values: 0


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

print("Feature scaling completed!")
print("Scaled data shape:", X_train_scaled.shape)

Feature scaling completed!
Scaled data shape: (79328, 22)


In [9]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [10]:
val_df = pd.read_csv("validation_fraud.csv")

print("Validation data loaded successfully!")
print("Validation shape:", val_df.shape)
print("\nClass distribution:")
print(val_df["is_fraud"].value_counts())

Validation data loaded successfully!
Validation shape: (8648, 45)

Class distribution:
is_fraud
0    8499
1     149
Name: count, dtype: int64


In [11]:
X_val = val_df[features]
y_val = val_df["is_fraud"]

print("Validation features prepared!")
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

Validation features prepared!
X_val shape: (8648, 22)
y_val shape: (8648,)


In [12]:
# Apply the training-fitted imputers to validation data

X_val[numeric_features] = numeric_imputer.transform(
    X_val[numeric_features]
)

X_val[binary_features] = binary_imputer.transform(
    X_val[binary_features]
)

print("Validation missing values handled!")
print("Total missing values:", X_val.isnull().sum().sum())

Validation missing values handled!
Total missing values: 0


In [13]:
X_val_scaled = scaler.transform(X_val)

print("Validation feature scaling completed!")
print("Scaled validation shape:", X_val_scaled.shape)

Validation feature scaling completed!
Scaled validation shape: (8648, 22)


In [14]:
print("Training feature count:", X_train_scaled.shape[1])
print("Validation feature count:", X_val_scaled.shape[1])

print("\nTraining features:")
print(X_train.columns.tolist())

print("\nValidation features:")
print(X_val.columns.tolist())

Training feature count: 22
Validation feature count: 22

Training features:
['amount', 'is_international', 'is_card_present', 'transactions_last_24h', 'average_amount_30d', 'risk_score', 'transaction_hour', 'transaction_day', 'transaction_month', 'sender_balance_before', 'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after', 'login_attempts', 'account_age_days', 'email_age_days', 'failed_payment_attempts', 'customer_balance', 'distance_from_home_km', 'transactions_last_1h', 'new_device', 'new_location']

Validation features:
['amount', 'is_international', 'is_card_present', 'transactions_last_24h', 'average_amount_30d', 'risk_score', 'transaction_hour', 'transaction_day', 'transaction_month', 'sender_balance_before', 'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after', 'login_attempts', 'account_age_days', 'email_age_days', 'failed_payment_attempts', 'customer_balance', 'distance_from_home_km', 'transactions_last_1h', 'new_device', 'new_loc

In [15]:
# Generate predictions
y_val_pred = logistic_model.predict(X_val_scaled)

# Generate fraud probabilities
y_val_prob = logistic_model.predict_proba(X_val_scaled)[:, 1]

print("Validation predictions generated successfully!")
print("Number of predictions:", len(y_val_pred))
print("Number of probabilities:", len(y_val_prob))

Validation predictions generated successfully!
Number of predictions: 8648
Number of probabilities: 8648


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
roc_auc = roc_auc_score(y_val, y_val_prob)

print("===== Logistic Regression Baseline =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

===== Logistic Regression Baseline =====
Accuracy : 0.6895
Precision: 0.0314
Recall   : 0.5705
F1 Score : 0.0595
ROC-AUC  : 0.6556


In [17]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, y_val_pred)

print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

Confusion Matrix:
[[5878 2621]
 [  64   85]]

True Negatives : 5878
False Positives: 2621
False Negatives: 64
True Positives : 85


In [18]:
results = pd.DataFrame({
    "Model": ["Logistic Regression Baseline"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1_Score": [f1],
    "ROC_AUC": [roc_auc],
    "True_Negatives": [tn],
    "False_Positives": [fp],
    "False_Negatives": [fn],
    "True_Positives": [tp]
})

results.to_csv("day10_logistic_regression_results.csv", index=False)

print("DAY 10 COMPLETED SUCCESSFULLY!")
print("\nResults:")
print(results)

DAY 10 COMPLETED SUCCESSFULLY!

Results:
                          Model  Accuracy  Precision   Recall  F1_Score  \
0  Logistic Regression Baseline  0.689524   0.031412  0.57047  0.059545   

    ROC_AUC  True_Negatives  False_Positives  False_Negatives  True_Positives  
0  0.655596            5878             2621               64              85  
